# Fundamentos de Forecasting en Python

Este cuaderno introduce los conceptos mas importantes de **forecasting** y muestra como comparar varios enfoques en Python con una metodologia consistente.

## Objetivos

- Entender tendencia, estacionalidad, rezagos y horizonte de pronostico.
- Crear una serie temporal reproducible para experimentar.
- Evaluar varios modelos con backtesting de ventana expansiva.
- Elegir el mejor modelo con metricas cuantitativas.
- Generar un forecast futuro con el modelo ganador.


## 1. Ideas clave de forecasting

Antes de modelar, conviene tener claras estas piezas:

- **Serie temporal**: observaciones ordenadas en el tiempo.
- **Tendencia**: movimiento ascendente o descendente de largo plazo.
- **Estacionalidad**: patrones que se repiten cada cierto periodo, por ejemplo cada 12 meses.
- **Ruido**: variacion aleatoria que el modelo no puede anticipar perfectamente.
- **Horizonte**: cuantos pasos hacia adelante queremos pronosticar.
- **Rezagos (lags)**: valores pasados que usamos como predictores.
- **Data leakage**: usar informacion del futuro sin querer. En forecasting debemos partir el dataset por tiempo, no aleatoriamente.
- **Backtesting**: evaluar el modelo simulando predicciones en distintos puntos del tiempo.

En este notebook vamos a comparar varios modelos bajo las mismas reglas para que la seleccion del mejor modelo sea justa.


In [ ]:
# Ejecuta esta celda solo si te faltan dependencias.
# %pip install -q numpy pandas matplotlib seaborn scikit-learn statsmodels prophet skforecast statsforecast


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX

PROPHET_AVAILABLE = True
PROPHET_IMPORT_ERROR = None
try:
    from prophet import Prophet
except Exception as exc:
    PROPHET_AVAILABLE = False
    PROPHET_IMPORT_ERROR = repr(exc)

SKFORECAST_AVAILABLE = True
SKFORECAST_IMPORT_ERROR = None
try:
    from skforecast.recursive import ForecasterRecursive
except Exception as exc:
    SKFORECAST_AVAILABLE = False
    SKFORECAST_IMPORT_ERROR = repr(exc)

STATSFORECAST_AVAILABLE = True
STATSFORECAST_IMPORT_ERROR = None
try:
    from statsforecast import StatsForecast
    from statsforecast.models import AutoARIMA
except Exception as exc:
    STATSFORECAST_AVAILABLE = False
    STATSFORECAST_IMPORT_ERROR = repr(exc)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (14, 5)

availability = pd.DataFrame(
    {
        "libreria": ["statsmodels", "prophet", "skforecast", "statsforecast"],
        "disponible": [True, PROPHET_AVAILABLE, SKFORECAST_AVAILABLE, STATSFORECAST_AVAILABLE],
        "detalle": [
            "incluida en imports principales",
            "ok" if PROPHET_AVAILABLE else PROPHET_IMPORT_ERROR,
            "ok" if SKFORECAST_AVAILABLE else SKFORECAST_IMPORT_ERROR,
            "ok" if STATSFORECAST_AVAILABLE else STATSFORECAST_IMPORT_ERROR,
        ],
    }
)

availability


## 2. Crear una serie temporal de ejemplo

Para que el cuaderno sea reproducible, construiremos una serie mensual con:

- tendencia creciente,
- estacionalidad anual,
- un pequeno impulso en noviembre y diciembre,
- ruido aleatorio.

Esto se parece a un caso de ventas mensuales, suficiente para practicar los conceptos fundamentales.


In [ ]:
dates = pd.date_range("2016-01-01", periods=120, freq="MS")
time = np.arange(len(dates))

trend = 80 + 0.9 * time
seasonality = 12 * np.sin(2 * np.pi * time / 12) + 4 * np.cos(2 * np.pi * time / 12)
holiday_boost = np.where(pd.Index(dates.month).isin([11, 12]), 10, 0)
noise = np.random.normal(loc=0, scale=4.5, size=len(dates))

sales = trend + seasonality + holiday_boost + noise

df = pd.DataFrame({"ventas": np.round(sales, 2)}, index=dates)
df.index.name = "fecha"
df.head()


In [ ]:
ax = df["ventas"].plot(color="#1f4e79", linewidth=2)
ax.set_title("Serie temporal mensual de ejemplo")
ax.set_ylabel("Ventas")
ax.set_xlabel("Fecha")
plt.show()


In [ ]:
decomposition = seasonal_decompose(df["ventas"], model="additive", period=12)
fig = decomposition.plot()
fig.set_size_inches(14, 10)
fig.suptitle("Descomposicion aditiva: tendencia, estacionalidad y residuo", y=1.02)
plt.show()


## 3. Separacion temporal y metricas

No usamos `train_test_split` aleatorio. En series temporales debemos respetar el orden cronologico.

Ademas, para elegir el mejor modelo no basta con una sola prediccion. Usaremos **backtesting con ventana expansiva**:

1. entrenamos con una parte inicial de la serie,
2. pronosticamos el siguiente horizonte,
3. avanzamos en el tiempo,
4. repetimos y promediamos las metricas.

Metricas usadas:

- **MAE**: error absoluto medio.
- **RMSE**: penaliza mas los errores grandes.
- **MAPE** y **sMAPE**: errores porcentuales para interpretar el desempeno relativo.


In [ ]:
SEASON_LENGTH = 12
HORIZON = 12
INITIAL_TRAIN_SIZE = 72
STEP = 12

train = df.iloc[:-HORIZON].copy()
test = df.iloc[-HORIZON:].copy()

print(f"Observaciones totales: {len(df)}")
print(f"Train final: {train.index.min().date()} -> {train.index.max().date()} ({len(train)} filas)")
print(f"Test holdout: {test.index.min().date()} -> {test.index.max().date()} ({len(test)} filas)")


In [ ]:
def infer_frequency(index: pd.DatetimeIndex) -> str:
    return index.freqstr or pd.infer_freq(index) or "MS"


def future_index(index: pd.DatetimeIndex, steps: int) -> pd.DatetimeIndex:
    freq = infer_frequency(index)
    return pd.date_range(start=index[-1], periods=steps + 1, freq=freq)[1:]


def compute_metrics(y_true: pd.Series, y_pred: pd.Series) -> dict:
    y_true = pd.Series(y_true, dtype=float)
    y_pred = pd.Series(y_pred, index=y_true.index, dtype=float)
    epsilon = 1e-8
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAPE (%)": np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), epsilon))) * 100,
        "sMAPE (%)": np.mean(
            2 * np.abs(y_pred - y_true) / np.maximum(np.abs(y_true) + np.abs(y_pred), epsilon)
        ) * 100,
    }


def summarize_metrics(results: pd.DataFrame) -> pd.DataFrame:
    metric_cols = ["MAE", "RMSE", "MAPE (%)", "sMAPE (%)"]
    summary = (
        results.groupby("modelo", as_index=True)[metric_cols]
        .mean()
        .sort_values(["MAE", "RMSE"])
    )
    return summary.round(3)


## 4. Funciones de forecast

Vamos a comparar cinco familias de modelos:

- **Naive estacional**: baseline muy importante. Repite el ultimo patron estacional.
- **`statsmodels` - Exponential Smoothing**: bueno para tendencia + estacionalidad.
- **`statsmodels` - SARIMAX**: clasico para series temporales con componentes autoregresivos y estacionales.
- **`prophet`**: modelo comodo para tendencia + estacionalidad + festivos.
- **`skforecast`**: convierte el forecasting en un problema supervisado usando rezagos.
- **`statsforecast`**: libreria optimizada para modelos estadisticos como `AutoARIMA`.

Si alguna libreria no esta instalada, el notebook la omitira y seguira con las disponibles.


In [ ]:
def seasonal_naive_forecast(train_series: pd.Series, steps: int, season_length: int = SEASON_LENGTH) -> pd.Series:
    if len(train_series) < season_length:
        raise ValueError("La serie de entrenamiento es demasiado corta para un naive estacional.")
    values = np.resize(train_series.iloc[-season_length:].to_numpy(), steps)
    return pd.Series(values, index=future_index(train_series.index, steps), name="yhat")


def ets_forecast(train_series: pd.Series, steps: int) -> pd.Series:
    model = ExponentialSmoothing(
        train_series,
        trend="add",
        seasonal="add",
        seasonal_periods=SEASON_LENGTH,
        initialization_method="estimated",
    )
    fitted = model.fit(optimized=True)
    pred = fitted.forecast(steps)
    pred.index = future_index(train_series.index, steps)
    return pred.rename("yhat")


def sarimax_forecast(train_series: pd.Series, steps: int) -> pd.Series:
    model = SARIMAX(
        train_series,
        order=(1, 1, 1),
        seasonal_order=(1, 1, 1, SEASON_LENGTH),
        trend="c",
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    fitted = model.fit(disp=False)
    pred = fitted.get_forecast(steps=steps).predicted_mean
    pred.index = future_index(train_series.index, steps)
    return pred.rename("yhat")


def prophet_forecast(train_series: pd.Series, steps: int) -> pd.Series:
    if not PROPHET_AVAILABLE:
        raise ImportError(f"Prophet no esta disponible: {PROPHET_IMPORT_ERROR}")
    prophet_df = train_series.reset_index()
    prophet_df.columns = ["ds", "y"]
    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        seasonality_mode="additive",
    )
    model.fit(prophet_df)
    freq = infer_frequency(train_series.index)
    future = model.make_future_dataframe(periods=steps, freq=freq)
    forecast = model.predict(future)
    pred = forecast.set_index("ds").iloc[-steps:]["yhat"]
    pred.index = future_index(train_series.index, steps)
    return pred.rename("yhat")


def skforecast_random_forest(train_series: pd.Series, steps: int) -> pd.Series:
    if not SKFORECAST_AVAILABLE:
        raise ImportError(f"skforecast no esta disponible: {SKFORECAST_IMPORT_ERROR}")
    forecaster = ForecasterRecursive(
        regressor=RandomForestRegressor(
            n_estimators=300,
            max_depth=6,
            min_samples_leaf=2,
            random_state=RANDOM_STATE,
        ),
        lags=SEASON_LENGTH,
    )
    forecaster.fit(y=train_series)
    pred = forecaster.predict(steps=steps)
    pred.index = future_index(train_series.index, steps)
    return pred.rename("yhat")


def statsforecast_autoarima(train_series: pd.Series, steps: int) -> pd.Series:
    if not STATSFORECAST_AVAILABLE:
        raise ImportError(f"statsforecast no esta disponible: {STATSFORECAST_IMPORT_ERROR}")
    freq = infer_frequency(train_series.index)
    train_df = pd.DataFrame(
        {
            "unique_id": "ventas",
            "ds": train_series.index,
            "y": train_series.values,
        }
    )
    sf = StatsForecast(models=[AutoARIMA(season_length=SEASON_LENGTH)], freq=freq, n_jobs=1)
    pred_df = sf.forecast(df=train_df, h=steps)
    model_columns = [col for col in pred_df.columns if col not in {"unique_id", "ds"}]
    pred = pred_df.set_index("ds")[model_columns[0]]
    pred.index = future_index(train_series.index, steps)
    return pred.rename("yhat")


## 5. Backtesting con ventana expansiva

El siguiente bloque evalua cada modelo varias veces. Asi evitamos tomar decisiones basadas en una sola particion afortunada.


In [ ]:
def backtest_expanding_window(
    series: pd.Series,
    forecast_fn,
    horizon: int = HORIZON,
    initial_train_size: int = INITIAL_TRAIN_SIZE,
    step: int = STEP,
):
    rows = []
    last_prediction_frame = None

    for train_end in range(initial_train_size, len(series) - horizon + 1, step):
        train_slice = series.iloc[:train_end]
        test_slice = series.iloc[train_end : train_end + horizon]
        predictions = forecast_fn(train_slice, horizon)
        predictions = pd.Series(predictions, index=test_slice.index, dtype=float)

        metric_values = compute_metrics(test_slice, predictions)
        rows.append(
            {
                "cutoff": train_slice.index[-1],
                **metric_values,
            }
        )

        last_prediction_frame = pd.DataFrame(
            {
                "real": test_slice,
                "predicho": predictions,
            }
        )

    return pd.DataFrame(rows), last_prediction_frame


model_registry = {
    "Naive estacional": seasonal_naive_forecast,
    "ETS (statsmodels)": ets_forecast,
    "SARIMAX (statsmodels)": sarimax_forecast,
    "Prophet": prophet_forecast,
    "RandomForest con lags (skforecast)": skforecast_random_forest,
    "AutoARIMA (statsforecast)": statsforecast_autoarima,
}

all_results = []
prediction_examples = {}
skipped_models = []

series = df["ventas"]

for model_name, model_fn in model_registry.items():
    try:
        bt_results, prediction_frame = backtest_expanding_window(series, model_fn)
        bt_results.insert(0, "modelo", model_name)
        all_results.append(bt_results)
        prediction_examples[model_name] = prediction_frame
    except Exception as exc:
        skipped_models.append({"modelo": model_name, "motivo": str(exc)})

results_df = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()
leaderboard = summarize_metrics(results_df) if not results_df.empty else pd.DataFrame()

leaderboard


In [ ]:
if skipped_models:
    skipped_df = pd.DataFrame(skipped_models)
    display(skipped_df)
else:
    print("Todos los modelos se evaluaron correctamente.")


### Como elegir el mejor modelo

Una forma practica de decidir es:

- ordenar por **MAE** o **RMSE**,
- revisar que el modelo sea estable a lo largo de varios cortes temporales,
- preferir el baseline solo si los modelos complejos no mejoran de forma clara,
- considerar costo de mantenimiento e interpretabilidad.

Aqui elegiremos el mejor segun el menor **MAE promedio**.


In [ ]:
if leaderboard.empty:
    raise RuntimeError("No se pudo evaluar ningun modelo. Revisa las dependencias disponibles.")

best_model_name = leaderboard.index[0]
best_model_fn = model_registry[best_model_name]

print(f"Mejor modelo segun MAE promedio: {best_model_name}")
display(results_df.groupby("modelo")[["MAE", "RMSE", "MAPE (%)", "sMAPE (%)"]].std().round(3))


In [ ]:
example_frame = prediction_examples[best_model_name]
ax = df["ventas"].iloc[-36:].plot(label="real", color="#1f4e79", linewidth=2)
example_frame["predicho"].plot(ax=ax, label=f"forecast - {best_model_name}", color="#d1495b", linewidth=2)
ax.set_title("Ultima ventana de backtesting del mejor modelo")
ax.set_ylabel("Ventas")
ax.legend()
plt.show()


## 6. Evaluacion final en holdout y forecast futuro

El backtesting ayuda a seleccionar el modelo. Despues podemos hacer dos cosas:

- medir el modelo ganador en un **holdout final**, y
- reentrenarlo con toda la serie disponible para pronosticar el futuro.


In [ ]:
holdout_pred = best_model_fn(train["ventas"], HORIZON)
holdout_pred.index = test.index
holdout_metrics = compute_metrics(test["ventas"], holdout_pred)

pd.DataFrame([holdout_metrics], index=[best_model_name]).round(3)


In [ ]:
future_forecast = best_model_fn(df["ventas"], HORIZON)
future_forecast.name = "forecast_futuro"
future_forecast.to_frame().head()


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
df["ventas"].iloc[-48:].plot(ax=ax, label="historico", color="#1f4e79", linewidth=2)
holdout_pred.plot(ax=ax, label="prediccion holdout", color="#d1495b", linewidth=2)
future_forecast.plot(ax=ax, label="forecast futuro", color="#2a9d8f", linewidth=2)
ax.axvline(test.index.min(), color="gray", linestyle="--", linewidth=1)
ax.set_title(f"Modelo ganador: {best_model_name}")
ax.set_ylabel("Ventas")
ax.legend()
plt.show()


## 7. Conclusiones

Este notebook deja una base practica para proyectos reales de forecasting:

- siempre compara contra un **baseline**,
- usa **particiones temporales** y no aleatorias,
- prefiere **backtesting** sobre una sola validacion,
- elige el modelo con mejor equilibrio entre error, estabilidad y complejidad,
- cuando tengas variables externas reales, incorporalas como regresores exogenos.

Siguientes mejoras posibles:

- agregar covariables como precio, promociones o dias festivos,
- probar mas hiperparametros para `SARIMAX`, `Prophet` o modelos basados en arboles,
- usar una serie real de tu negocio y repetir exactamente la misma logica de evaluacion.
